In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

np.random.seed(42)
n_samples = 100

usia = np.random.randint(21, 60, size=n_samples)
pendapatan = np.random.uniform(3.0, 30.0, size=n_samples).round(2)  # dalam juta rupiah
skor_kredit = np.random.randint(300, 850, size=n_samples)
tanggungan = np.random.randint(0, 5, size=n_samples)

# Logika penentuan status pinjaman: dipengaruhi pendapatan dan skor kredit
skor_total = (pendapatan * 15) + (skor_kredit * 0.5) - (tanggungan * 40) + np.random.normal(0, 30, n_samples)
status = np.where(skor_total > 350, 'Diterima', 'Ditolak')

df_loan = pd.DataFrame({
    'Usia': usia,
    'Pendapatan_Juta': pendapatan,
    'Skor_Kredit': skor_kredit,
    'Jumlah_Tanggungan': tanggungan,
    'Status_Pinjaman': status
})

# Tampilkan 5 baris pertama dan distribusi kelas
print("Preview Dataset:")
print(df_loan.head())
print("\nDistribusi Kelas Target:")
print(df_loan['Status_Pinjaman'].value_counts())

X = df_loan[['Usia', 'Pendapatan_Juta', 'Skor_Kredit', 'Jumlah_Tanggungan']]
y = df_loan['Status_Pinjaman']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Standardisasi fitur untuk k-NN (skala fitur sangat timpang)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# A. Naive Bayes (Gaussian)
nb = GaussianNB()
nb.fit(X_train, y_train)  # NB tidak wajib scaling
nb_pred = nb.predict(X_test)
acc_nb = accuracy_score(y_test, nb_pred)

# B. k-NN (k=5 dengan fitur terstandardisasi)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, knn_pred)

print("\n" + "="*40)
print(f"Akurasi Naive Bayes (Gaussian) : {acc_nb:.4f}")
print(f"Akurasi k-NN (k=5)             : {acc_knn:.4f}")
print("="*40)

print("\n--- Classification Report: Naive Bayes ---")
print(classification_report(y_test, nb_pred, digits=3))

print("--- Classification Report: k-NN (k=5) ---")
print(classification_report(y_test, knn_pred, digits=3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, nb_pred, 
    cmap='Blues', 
    ax=axes[0], 
    colorbar=False
)
axes[0].set_title(f'Naive Bayes (Akurasi: {acc_nb:.2%})', fontweight='bold')
axes[0].grid(False)

ConfusionMatrixDisplay.from_predictions(
    y_test, knn_pred, 
    cmap='Blues', 
    ax=axes[1], 
    colorbar=False
)
axes[1].set_title(f'k-NN (k=5) (Akurasi: {acc_knn:.2%})', fontweight='bold')
axes[1].grid(False)

plt.suptitle('Perbandingan Confusion Matrix pada Dataset Kelayakan Pinjaman', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()